In [1]:
import pandas as pd

In [2]:
stroke_data = pd.read_csv('../../data/raw/healthcare-dataset-stroke-data.csv')
stroke_data

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0


In [3]:
# Data cleaning (BMI has missing values; replacing them with the median)
stroke_data['bmi'] = stroke_data['bmi'].fillna(stroke_data['bmi'].median())

In [4]:
X = stroke_data.drop(columns = ['id','stroke'])
Y = stroke_data[['stroke']]

In [5]:
from sklearn.model_selection import train_test_split
#Train Test split
X0_train, X0_test, Y0_train, Y0_test = train_test_split(X, Y, test_size=0.2, random_state=42)

#Flatten y
Y0_train = Y0_train.values.ravel()
Y0_test = Y0_test.values.ravel()

In [6]:
num_cols = ["age", "avg_glucose_level", "bmi","hypertension", "heart_disease"]
cat_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]



In [14]:
df_stroke_yes = stroke_data.loc[stroke_data["stroke"]== 1, num_cols+cat_cols]
df_stroke_yes

,age,avg_glucose_level,bmi,hypertension,heart_disease,gender,ever_married,work_type,Residence_type,smoking_status
0,67.0,228.69,36.6,0,1,Male,Yes,Private,Urban,formerly smoked
1,61.0,202.21,28.1,0,0,Female,Yes,Self-employed,Rural,never smoked
2,80.0,105.92,32.5,0,1,Male,Yes,Private,Rural,never smoked
3,49.0,171.23,34.4,0,0,Female,Yes,Private,Urban,smokes
4,79.0,174.12,24.0,1,0,Female,Yes,Self-employed,Rural,never smoked
...,...,...,...,...,...,...,...,...,...,...
244,57.0,84.96,36.7,0,0,Male,Yes,Private,Rural,Unknown
245,14.0,57.93,30.9,0,0,Female,No,children,Rural,Unknown
246,75.0,78.80,29.3,0,0,Female,Yes,Self-employed,Rural,formerly smoked
247,71.0,87.80,28.1,1,0,Male,Yes,Self-employed,Rural,Unknown


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
#Define preprocessing
preproc1 = ColumnTransformer(
    transformers=[
    ("nums", StandardScaler(), num_cols),
    ("cats", OneHotEncoder(handle_unknown="ignore"),cat_cols)
],
     remainder='passthrough'

)

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso

# Pipeline A = preproc1 + baseline
pipeA = Pipeline(
    [
        ('preprocessing', preproc1),
        ('regressor', Lasso(max_iter=10000))

    ]
)

In [9]:
from sklearn.ensemble import HistGradientBoostingRegressor

# Pipeline B = preproc1 + advanced model
pipeB = Pipeline(
    [
        ('preprocessing', preproc1),
        ('regressor', HistGradientBoostingRegressor(random_state=42))
    ]
)

In [10]:
from sklearn.model_selection import GridSearchCV
#Gridsearch
param_grid_A = {
    'regressor__alpha':[0.01, 0.1, 0.5, 1, 5]
    }
gridcv_A = GridSearchCV(
    pipeA,
    param_grid_A,
    scoring = 'neg_mean_absolute_error',
    cv = 5
)
gridcv_A.fit(X0_train, Y0_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...iter=10000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'regressor__alpha': [0.01, 0.1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >

In [11]:
param_grid_B = {
    "regressor__max_depth": [3, 5],
    "regressor__learning_rate": [0.05, 0.1],
    "regressor__max_iter": [200, 400]
    }
gridcv_B = GridSearchCV(
    pipeB,
    param_grid_B,
    scoring = 'neg_mean_absolute_error',
    cv = 5,
    n_jobs=-1
)
gridcv_B.fit(X0_train, Y0_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'regressor__learning_rate': [0.05, 0.1], 'regressor__max_depth': [3, 5], 'regressor__max_iter': [200, 400]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and par

In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [13]:
import numpy as np

def evaluate(grid, name):
    model = grid.best_estimator_
    preds = model.predict(X0_test)

    mae = mean_absolute_error(Y0_test, preds)
    rmse = np.sqrt(mean_squared_error(Y0_test, preds))
    r2 = r2_score(Y0_test, preds)

    print(name)
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2  :", r2)
    print("----------------")

    return mae

In [27]:
mae_A = evaluate(gridcv_A, "Pipeline A")
mae_B = evaluate(gridcv_B, "Pipeline B")

Pipeline A
MAE : 0.10085889300362667
RMSE: 0.2391813823757876
R2  : -0.003907300067204522
----------------
Pipeline B
MAE : 0.09132851319437332
RMSE: 0.2283192174507898
R2  : 0.08520495501088998
----------------
